# Where Is Waldo? End-to-end Colab workflow

Run this notebook with **Runtime > Change runtime type > T4 GPU**. It executes the same scripts documented in the repository, so notebook and command-line results remain consistent. Do not upload API keys or raw data to GitHub.

In [ ]:
!nvidia-smi
from pathlib import Path
REPO_URL = 'https://github.com/ACM40960/projects-caixuan-swathi.git'
PROJECT = Path('/content/projects-caixuan-swathi')
if not PROJECT.exists():
    !git clone {REPO_URL} {PROJECT}
%cd /content/projects-caixuan-swathi
!pip install -q -e '.[train,dev]'

## Provide the dataset

Upload the YOLOv8 ZIP downloaded from Wally-Finder v5. The archive is extracted locally in the temporary Colab runtime and is not committed.

In [ ]:
from google.colab import files
import shutil
uploaded = files.upload()
zip_name = next(name for name in uploaded if name.lower().endswith('.zip'))
DATASET = Path('/content/wally-finder-v5')
DATASET.mkdir(parents=True, exist_ok=True)
shutil.unpack_archive(zip_name, DATASET)
print('Dataset YAML files:', list(DATASET.rglob('*.yaml')))

## 1. Data audit

Stop if the audit reports cross-split exact or perceptual duplicates. Resolve leakage before training.

In [ ]:
!python scripts/audit_dataset.py --dataset {DATASET} --output artifacts/audit_original
GROUPED = Path('/content/wally-finder-grouped')
!python scripts/create_grouped_split.py --dataset {DATASET} --output {GROUPED} --seed 42
!python scripts/audit_dataset.py --dataset {GROUPED} --output artifacts/audit_grouped
DATASET = GROUPED

## 2. Template-matching baseline

In [ ]:
!python scripts/export_ground_truth.py --dataset {DATASET} --split test --output artifacts/test_ground_truth.csv
!python scripts/run_baseline.py --dataset {DATASET} --split test --output artifacts/baseline
!python scripts/evaluate_predictions.py --dataset {DATASET} --truth artifacts/test_ground_truth.csv --predictions artifacts/baseline/predictions_test.csv --output artifacts/evaluation/template

## 3. Full-page YOLOv8n baseline

The validation set controls early stopping. The test set remains untouched until prediction.

In [ ]:
DATA_YAML = next(DATASET.rglob('*.yaml'))
!python scripts/train_yolo.py --data {DATA_YAML} --name yolov8n_full --epochs 80 --imgsz 512 --batch 16 --device 0
TEST_IMAGES = next(path for path in DATASET.rglob('test/images') if path.is_dir())
!python scripts/predict_yolo.py --weights runs/detect/yolov8n_full/weights/best.pt --images {TEST_IMAGES} --output artifacts/yolov8n_full_predictions.csv --imgsz 512 --device 0
!python scripts/evaluate_predictions.py --dataset {DATASET} --truth artifacts/test_ground_truth.csv --predictions artifacts/yolov8n_full_predictions.csv --output artifacts/evaluation/yolov8n_full

## 4. Overlapping tiles and tiled YOLOv8n

Images are split first; tiles never cross source splits. Test-tile detections are translated back to page coordinates and class-aware NMS removes overlap duplicates.

In [ ]:
TILED = Path('/content/wally-finder-tiles')
!python scripts/slice_dataset.py --dataset {DATASET} --output {TILED} --tile-size 256 --overlap 64 --negative-ratio 3 --seed 42
!python scripts/train_yolo.py --data {TILED / 'data.yaml'} --name yolov8n_tiles --epochs 100 --imgsz 512 --batch 16 --device 0
!python scripts/predict_tiled.py --weights runs/detect/yolov8n_tiles/weights/best.pt --tiled-dataset {TILED} --output artifacts/yolov8n_tiles_predictions.csv --imgsz 512 --device 0
!python scripts/evaluate_predictions.py --dataset {DATASET} --truth artifacts/test_ground_truth.csv --predictions artifacts/yolov8n_tiles_predictions.csv --output artifacts/evaluation/yolov8n_tiles

## 5. Error-analysis figures and saved outputs

In [ ]:
!python scripts/plot_error_analysis.py \
  --metrics template=artifacts/evaluation/template/per_class_metrics.csv full=artifacts/evaluation/yolov8n_full/per_class_metrics.csv tiles=artifacts/evaluation/yolov8n_tiles/per_class_metrics.csv \
  --errors template=artifacts/evaluation/template/matched_predictions_and_errors.csv full=artifacts/evaluation/yolov8n_full/matched_predictions_and_errors.csv tiles=artifacts/evaluation/yolov8n_tiles/matched_predictions_and_errors.csv \
  --output artifacts/figures
!zip -qr /content/waldo_results.zip artifacts runs/detect/*/results.csv runs/detect/*/args.yaml runs/detect/*/weights/best.pt
files.download('/content/waldo_results.zip')